In [1]:
from TDScoringAgent import TDScoringAgent
from TDAgentBase import TDAgentEnum, TDAgentState
from intent_gate import create_intent_capsule, SecurityError
import time
import json

In [2]:
# TEST CASE 1: VALID INTENT CAPSULE (Scoring Agent)


def test_case_1_valid_capsule():
    """
    Test Case 1: Valid Intent Capsule

    Shows: create_intent_capsule() works for Scoring Agent
    """
    print("\n" + "=" * 70)
    print("TEST CASE 1: VALID INTENT CAPSULE (Scoring Agent)")
    print("=" * 70 + "\n")

    secret_key = "dev_secret"
    scoring_agent = TDScoringAgent(
        agent_id=TDAgentEnum.SCORING_AGENT, secret_key=secret_key
    )
    print("✓ Step 1: Created TDScoringAgent")
    print()

    # Create capsule with flexible parameters
    trip_id = "TRIP-001"
    intent_capsule = create_intent_capsule(
        trip_id=trip_id,
        secret_key=secret_key,
        expires_at=time.time() + 60,
        subject="scoring_agent",
        purpose="evaluate_trip_lifecycle",
        allowed_actions=["score_trip"],
    )
    print("✓ Step 2: Orchestrator created Intent Capsule")
    print(f"  Trip ID: {trip_id}")
    print(f"  Subject: {intent_capsule['capsule']['subject']}")
    print(f"  Purpose: {intent_capsule['capsule']['purpose']}")
    print(
        f"  Allowed Actions: {intent_capsule['capsule']['constraints']['allowed_actions']}"
    )
    print()

    trip_data: TDAgentState = {
        "trip_id": trip_id,
        "trip_context": {
            "distance_km": 40.3,
            "harsh_events": 3,
        },
        "intent_capsule": intent_capsule,
    }
    print("✓ Step 3: Created state with trip_context + Intent Capsule")
    print()

    try:
        result = scoring_agent.run(trip_data)
        print("✓ Step 4: Intent Gate validated capsule")
        print("  - Signature valid")
        print("  - Not expired")
        print("  - Gate PASSED")
        print()
        print("✓ Step 5: Agent executed successfully")
        print(f"  Trip Score: {result['output']['trip_score']}/100")

    except SecurityError as e:
        print(f"✗ FAILED: {e}")


test_case_1_valid_capsule()


TEST CASE 1: VALID INTENT CAPSULE (Scoring Agent)

✓ Step 1: Created TDScoringAgent

✓ Step 2: Orchestrator created Intent Capsule
  Trip ID: TRIP-001
  Subject: scoring_agent
  Purpose: evaluate_trip_lifecycle
  Allowed Actions: ['score_trip']

✓ Step 3: Created state with trip_context + Intent Capsule

✓ Step 4: Intent Gate validated capsule
  - Signature valid
  - Not expired
  - Gate PASSED

✓ Step 5: Agent executed successfully
  Trip Score: 85/100


In [3]:
test_case_1_valid_capsule()


TEST CASE 1: VALID INTENT CAPSULE (Scoring Agent)

✓ Step 1: Created TDScoringAgent

✓ Step 2: Orchestrator created Intent Capsule
  Trip ID: TRIP-001
  Subject: scoring_agent
  Purpose: evaluate_trip_lifecycle
  Allowed Actions: ['score_trip']

✓ Step 3: Created state with trip_context + Intent Capsule

✓ Step 4: Intent Gate validated capsule
  - Signature valid
  - Not expired
  - Gate PASSED

✓ Step 5: Agent executed successfully
  Trip Score: 85/100


In [4]:
# TEST CASE 2: EXPIRED INTENT CAPSULE (Safety Agent)


def test_case_2_expired_capsule():
    """
    Test Case 2: Expired Intent Capsule

    Shows: create_intent_capsule() works for Safety Agent
            with custom purpose and allowed_actions
    """
    print("\n" + "=" * 70)
    print("TEST CASE 2: EXPIRED INTENT CAPSULE (Safety Agent)")
    print("=" * 70 + "\n")

    secret_key = "dev_secret"
    safety_agent = (
        TDScoringAgent(  # Using ScoringAgent as placeholder (same base class)
            agent_id=TDAgentEnum.SAFETY_AGENT, secret_key=secret_key
        )
    )
    print("✓ Step 1: Created Safety Agent")
    print()

    # Create capsule with Safety Agent parameters
    trip_id = "TRIP-002"
    intent_capsule = create_intent_capsule(
        trip_id=trip_id,
        secret_key=secret_key,
        expires_at=time.time() - 10,  # EXPIRED 10 seconds ago
        subject="safety_agent",
        purpose="enrich_harsh_events",
        allowed_actions=["enrich_events"],
    )
    print("✓ Step 2: Orchestrator created Intent Capsule (EXPIRED)")
    print(f"  Trip ID: {trip_id}")
    print(f"  Subject: {intent_capsule['capsule']['subject']}")
    print(f"  Purpose: {intent_capsule['capsule']['purpose']}")
    print(
        f"  Allowed Actions: {intent_capsule['capsule']['constraints']['allowed_actions']}"
    )
    print(f"  Expired at: {int(intent_capsule['capsule']['expires_at'])}")
    print(f"  Current time: {int(time.time())}")
    print()

    trip_data: TDAgentState = {
        "trip_id": trip_id,
        "trip_context": {
            "distance_km": 35.0,
            "harsh_events": 2,
        },
        "intent_capsule": intent_capsule,
    }
    print("✓ Step 3: Created state with EXPIRED Intent Capsule")
    print()

    try:
        result = safety_agent.run(trip_data)
        print("✗ FAILED: Agent should have been blocked!")

    except SecurityError as e:
        print("✓ Step 4: Intent Gate checked expiration")
        print("  - Capsule is EXPIRED")
        print("  - Gate REJECTED")
        print()
        print(f"Error: {e}")


test_case_2_expired_capsule()

Capsule expired for trip TRIP-002. Valid until: 1774480002, Current: 1774480012.513971



TEST CASE 2: EXPIRED INTENT CAPSULE (Safety Agent)

✓ Step 1: Created Safety Agent

✓ Step 2: Orchestrator created Intent Capsule (EXPIRED)
  Trip ID: TRIP-002
  Subject: safety_agent
  Purpose: enrich_harsh_events
  Allowed Actions: ['enrich_events']
  Expired at: 1774480002
  Current time: 1774480012

✓ Step 3: Created state with EXPIRED Intent Capsule

✓ Step 4: Intent Gate checked expiration
  - Capsule is EXPIRED
  - Gate REJECTED

Error: Intent Capsule EXPIRED. Valid until: 1774480002, Current: 1774480012.513971


In [5]:
# TEST CASE 3: TAMPERED INTENT CAPSULE (Support Agent)


def test_case_3_tampered_capsule():
    """
    Test Case 3: Tampered Intent Capsule

    Shows: create_intent_capsule() works for Support Agent
            with custom constraints
    """
    print("\n" + "=" * 70)
    print("TEST CASE 3: TAMPERED INTENT CAPSULE (Support Agent)")
    print("=" * 70 + "\n")

    secret_key = "dev_secret"
    support_agent = (
        TDScoringAgent(  # Using ScoringAgent as placeholder (same base class)
            agent_id=TDAgentEnum.SCORING_AGENT, secret_key=secret_key  # Reuse for demo
        )
    )
    print("✓ Step 1: Created Support Agent")
    print()

    # Create capsule with Support Agent parameters
    trip_id = "TRIP-003"
    intent_capsule = create_intent_capsule(
        trip_id=trip_id,
        secret_key=secret_key,
        expires_at=time.time() + 60,
        subject="support_agent",
        purpose="generate_coaching_tips",
        allowed_actions=["generate_tips", "send_notification"],
    )
    original_signature = intent_capsule["signature"]
    print("✓ Step 2: Orchestrator created Intent Capsule (VALID)")
    print(f"  Trip ID: {trip_id}")
    print(f"  Subject: {intent_capsule['capsule']['subject']}")
    print(f"  Purpose: {intent_capsule['capsule']['purpose']}")
    print(
        f"  Allowed Actions: {intent_capsule['capsule']['constraints']['allowed_actions']}"
    )
    print(f"  Original signature: {original_signature[:20]}...")
    print()

    # ATTACK: Tamper with allowed_actions
    print("✓ Step 3: Attacker tampers with capsule")
    intent_capsule["capsule"]["constraints"]["allowed_actions"] = ["delete_everything"]
    print(f"  Changed: allowed_actions → ['delete_everything']")
    print(f"  (Signature is now INVALID)")
    print()

    trip_data: TDAgentState = {
        "trip_id": trip_id,
        "trip_context": {
            "distance_km": 45.0,
            "harsh_events": 1,
        },
        "intent_capsule": intent_capsule,
    }
    print("✓ Step 4: Created state with TAMPERED Intent Capsule")
    print()

    try:
        result = support_agent.run(trip_data)
        print("✗ FAILED: Agent should have been blocked!")

    except SecurityError as e:
        print("✓ Step 5: Intent Gate verified signature")
        print("  - Signature MISMATCH detected")
        print("  - Gate REJECTED")
        print()
        print(f"Error: {e}")


test_case_3_tampered_capsule()

Tampering detected for trip TRIP-003. Signature mismatch.



TEST CASE 3: TAMPERED INTENT CAPSULE (Support Agent)

✓ Step 1: Created Support Agent

✓ Step 2: Orchestrator created Intent Capsule (VALID)
  Trip ID: TRIP-003
  Subject: support_agent
  Purpose: generate_coaching_tips
  Allowed Actions: ['generate_tips', 'send_notification']
  Original signature: 04900261da0f679688fb...

✓ Step 3: Attacker tampers with capsule
  Changed: allowed_actions → ['delete_everything']
  (Signature is now INVALID)

✓ Step 4: Created state with TAMPERED Intent Capsule

✓ Step 5: Intent Gate verified signature
  - Signature MISMATCH detected
  - Gate REJECTED

Error: Intent Capsule TAMPERING DETECTED. Signature mismatch.
